# ETL Cripto: Extract → Transform → Load (múltiples tablas)

Versión mejorada de tu notebook original. Cambios principales:

1. **E, T y L quedan en secciones separadas y marcadas** — antes la llamada a NewsAPI
   vivía dentro de `transform_data()`; ahora es su propia extracción (una segunda ronda,
   porque depende de una decisión tomada en el primer Transform: quiénes son los "top movers").
2. **El Load ya no es una sola tabla** (`crypto_history`) — son **4 tablas** con su propia
   llave primaria, pensadas como un mini modelo dimensional:
   - `dim_moneda_cripto` — catálogo de monedas (id, symbol, name). No repite en cada fila.
   - `hechos_mercado_cripto` — un snapshot de precio/volumen/cambio % por moneda y momento.
   - `noticias_cripto` — solo las monedas que sí tuvieron una noticia real (no placeholders).
   - `tipo_cambio` — la misma tabla del ejercicio de tipo de cambio USD/MXN; aquí se le
     agrega el punto de USD/EUR del día. Si ya corriste ese ejercicio contra la misma base,
     esta carga convive con esos datos sin pisarlos.
3. **Credenciales fuera del código** — NewsAPI key y password de MariaDB se piden en
   tiempo de ejecución (`getpass` o Colab Secrets), no quedan escritas en el notebook.

> ⚠️ Si ya corriste el notebook original y compartiste el link: la NewsAPI key y el
> password de MariaDB que tenía escritos deberían rotarse — quedaron expuestos en texto plano.

## 0. Instalar dependencias e importar

In [ ]:
!pip install -q pymysql sqlalchemy

import time
import json
from datetime import datetime, date, timezone
from pathlib import Path

import requests as req
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

## 1. Credenciales (fuera del código)

Se intenta primero con **Colab Secrets** (ícono de llave 🔑 en el panel izquierdo — más
seguro porque no se piden cada vez que se corre el notebook). Si no existen ahí, se piden
por `getpass` como respaldo.

In [ ]:
from getpass import getpass

def obtener_secreto(nombre, prompt):
    try:
        from google.colab import userdata
        valor = userdata.get(nombre)
        if valor:
            return valor
    except Exception:
        pass
    return getpass(prompt)

NEWS_API_KEY = obtener_secreto("NEWS_API_KEY", "Pega tu NewsAPI key: ")
DB_HOST = obtener_secreto("DB_HOST", "IP de tu instancia de MariaDB: ")
DB_USER = obtener_secreto("DB_USER", "Usuario de MariaDB: ")
DB_PASSWORD = obtener_secreto("DB_PASSWORD", "Password de MariaDB: ")
DB_NAME = "prices"
DB_PORT = 3306

---
# 🟦 EXTRACT (ronda 1) — mercado y tipo de cambio

Dos llamadas a APIs públicas, sin transformar nada todavía. Ambas son la capa "bronze".

In [ ]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)


def extract_exchange_rate(moneda_destino="EUR"):
    """Tipo de cambio USD -> moneda_destino, vía Frankfurter (sin API key)."""
    url = "https://api.frankfurter.dev/v1/latest"
    resp = req.get(url, params={"base": "USD", "symbols": moneda_destino}, timeout=10)
    resp.raise_for_status()
    return resp.json()


def extract_market_snapshot(per_page=30):
    """Top `per_page` criptomonedas por market cap, vía CoinGecko (sin API key)."""
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": per_page,
        "price_change_percentage": "24h",
    }
    resp = req.get(url, params=params, timeout=10)
    resp.raise_for_status()
    return resp.json()


def guardar_crudo(datos, nombre):
    ruta = RAW_DIR / f"{nombre}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S}.json"
    with open(ruta, "w", encoding="utf-8") as f:
        json.dump(datos, f, ensure_ascii=False, indent=2)
    return ruta

In [ ]:
raw_fx = extract_exchange_rate("EUR")
raw_market = extract_market_snapshot(per_page=30)

guardar_crudo(raw_fx, "tipo_cambio")
guardar_crudo(raw_market, "mercado_cripto")

eur_rate = raw_fx["rates"]["EUR"]
print(f"EXTRACT OK — tipo de cambio USD/EUR: {eur_rate} | {len(raw_market)} monedas extraídas")

---
# 🟨 TRANSFORM (ronda 1) — limpiar mercado y decidir "top movers"

Aquí se limpia el JSON crudo de CoinGecko y se calculan las columnas derivadas. **Esta es
también la etapa que decide para cuáles monedas vale la pena gastar cuota de NewsAPI** —
los 5 que más subieron y los 5 que más bajaron en 24h. Esa decisión es la que conecta con
el segundo Extract de abajo.

In [ ]:
def transform_market_data(raw_list, eur_rate, fecha_snapshot):
    """Limpia el JSON de CoinGecko y calcula columnas derivadas. No llama a ninguna API."""
    if not raw_list:
        return pd.DataFrame(), pd.DataFrame(), []

    df = pd.DataFrame(raw_list)

    cols_map = {
        "id": "coin_id", "symbol": "symbol", "name": "name",
        "current_price": "price_usd", "price_change_percentage_24h": "change_24h_pct",
        "total_volume": "volume_24h_usd", "market_cap": "market_cap_usd",
    }
    df_clean = df[list(cols_map.keys())].rename(columns=cols_map).copy()

    # --- Tabla de dimensión (catálogo de monedas, sin duplicar en cada snapshot) ---
    df_dim_moneda = df_clean[["coin_id", "symbol", "name"]].drop_duplicates()

    # --- Tabla de hechos (el snapshot de mercado de este momento) ---
    df_hechos = df_clean.drop(columns=["symbol", "name"]).copy()
    df_hechos["fecha_snapshot"] = fecha_snapshot
    df_hechos["price_eur"] = df_hechos["price_usd"] * eur_rate
    df_hechos["is_alert"] = df_hechos["change_24h_pct"].apply(lambda x: abs(x or 0) >= 5)

    # --- Decisión que alimenta el segundo Extract: quiénes son "top movers" ---
    top_movers_ids = pd.concat([
        df_clean.nlargest(5, "change_24h_pct"),
        df_clean.nsmallest(5, "change_24h_pct"),
    ])["coin_id"].unique().tolist()

    return df_dim_moneda, df_hechos, top_movers_ids

In [ ]:
fecha_snapshot = datetime.now(timezone.utc).replace(second=0, microsecond=0, tzinfo=None)

df_dim_moneda, df_hechos_mercado, top_movers_ids = transform_market_data(
    raw_market, eur_rate, fecha_snapshot
)

print(f"TRANSFORM OK — {len(df_hechos_mercado)} monedas limpias, "
      f"{len(top_movers_ids)} identificadas como top movers")
df_hechos_mercado.head()

---
# 🟦 EXTRACT (ronda 2) — noticias, solo para los top movers

Esta llamada **no existiría sin el Transform anterior**: solo sabemos a quién preguntarle
a NewsAPI porque el paso de arriba calculó quiénes se movieron más. Es un patrón normal en
pipelines de enriquecimiento (Extract → Transform → Extract), y a propósito se deja
como su propia sección para que no se confunda con el Transform.

In [ ]:
def extract_news_for_coin(coin_name):
    """Un artículo reciente para `coin_name`. Devuelve (titulo, url) o (None, None)."""
    url = "https://newsapi.org/v2/everything"
    params = {
        "q": f"{coin_name} crypto",
        "sortBy": "publishedAt",
        "pageSize": 1,
        "apiKey": NEWS_API_KEY,
    }
    try:
        resp = req.get(url, params=params, timeout=5)
        data = resp.json()
        if data.get("status") == "ok" and data.get("articles"):
            art = data["articles"][0]
            return art["title"], art["url"]
    except req.exceptions.RequestException as e:
        print(f"  aviso: no se pudo consultar noticias de {coin_name}: {e}")
    return None, None

In [ ]:
nombres_top_movers = df_dim_moneda[
    df_dim_moneda["coin_id"].isin(top_movers_ids)
][["coin_id", "name"]]

raw_noticias = []
for _, fila in nombres_top_movers.iterrows():
    titulo, link = extract_news_for_coin(fila["name"])
    raw_noticias.append({"coin_id": fila["coin_id"], "news_title": titulo, "news_url": link})
    time.sleep(0.2)  # cuidar la cuota gratuita de NewsAPI

guardar_crudo(raw_noticias, "noticias_cripto")
print(f"EXTRACT OK — {len(raw_noticias)} monedas consultadas en NewsAPI")

---
# 🟨 TRANSFORM (ronda 2) — tabla de noticias + validación de calidad

A diferencia del notebook original (que guardaba una fila de "Estable (sin noticias
clave)" para cada moneda estable), aquí **solo se guardan las monedas que sí tuvieron un
artículo real**. Las que no tuvieron noticia, simplemente no aparecen en la tabla — es un
diseño de base de datos más limpio que llenarla de filas placeholder.

In [ ]:
def transform_noticias(raw_noticias, fecha_snapshot):
    df = pd.DataFrame(raw_noticias)
    df = df.dropna(subset=["news_title", "news_url"])  # descartar donde no hubo artículo real
    df["fecha_snapshot"] = fecha_snapshot
    return df[["coin_id", "fecha_snapshot", "news_title", "news_url"]]


def transform_tipo_cambio(raw_fx):
    fecha = pd.to_datetime(raw_fx["date"]).date()
    filas = [
        {"fecha": fecha, "moneda_origen": raw_fx["base"], "moneda_destino": moneda, "tasa": tasa}
        for moneda, tasa in raw_fx["rates"].items()
    ]
    return pd.DataFrame(filas)


df_noticias = transform_noticias(raw_noticias, fecha_snapshot)
df_tipo_cambio = transform_tipo_cambio(raw_fx)

print(f"TRANSFORM OK — {len(df_noticias)} noticias reales (de {len(raw_noticias)} monedas consultadas)")
df_noticias

In [ ]:
def validar_calidad(df_hechos, df_tipo_cambio):
    errores = []

    if df_hechos["price_usd"].isnull().any():
        errores.append("Hay precios nulos en hechos_mercado.")
    if (df_hechos["price_usd"] < 0).any():
        errores.append("Hay precios negativos en hechos_mercado.")
    if df_hechos.duplicated(subset=["coin_id", "fecha_snapshot"]).any():
        errores.append("Hay filas duplicadas en hechos_mercado por (coin_id, fecha_snapshot).")
    if (df_tipo_cambio["tasa"] <= 0).any():
        errores.append("Hay una tasa de cambio <= 0.")

    if errores:
        for e in errores:
            print(f"  - {e}")
        raise ValueError("Los datos no pasaron la validación de calidad.")
    print("Validación de calidad: OK")


validar_calidad(df_hechos_mercado, df_tipo_cambio)

---
# 🟩 LOAD — 4 tablas, con upsert genérico

`upsert_dataframe()` es una sola función reusada para las 4 tablas: cada tabla ya definió
su propia llave primaria en `schema.sql`, así que la función solo necesita el nombre de la
tabla y el DataFrame — no hay que repetir la lógica de upsert 4 veces.

In [ ]:
DB_URL = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
ENGINE = create_engine(DB_URL)

SQL_CREATE_TABLAS = """
CREATE TABLE IF NOT EXISTS dim_moneda_cripto (
    coin_id VARCHAR(50) NOT NULL,
    symbol  VARCHAR(20) NOT NULL,
    name    VARCHAR(100) NOT NULL,
    PRIMARY KEY (coin_id)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS hechos_mercado_cripto (
    coin_id        VARCHAR(50)   NOT NULL,
    fecha_snapshot DATETIME      NOT NULL,
    price_usd      DECIMAL(18,6),
    price_eur      DECIMAL(18,6),
    volume_24h_usd DECIMAL(24,2),
    market_cap_usd DECIMAL(24,2),
    change_24h_pct DECIMAL(8,4),
    is_alert       TINYINT(1),
    PRIMARY KEY (coin_id, fecha_snapshot)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS noticias_cripto (
    coin_id        VARCHAR(50)  NOT NULL,
    fecha_snapshot DATETIME     NOT NULL,
    news_title     VARCHAR(500),
    news_url       VARCHAR(500),
    PRIMARY KEY (coin_id, fecha_snapshot)
) ENGINE=InnoDB;

CREATE TABLE IF NOT EXISTS tipo_cambio (
    fecha             DATE           NOT NULL,
    moneda_origen     VARCHAR(3)     NOT NULL,
    moneda_destino    VARCHAR(3)     NOT NULL,
    tasa              DECIMAL(14,6)  NOT NULL,
    variacion_diaria  DECIMAL(14,6)  NULL,
    variacion_pct     DECIMAL(8,4)   NULL,
    promedio_movil_7d DECIMAL(14,6)  NULL,
    PRIMARY KEY (fecha, moneda_origen, moneda_destino)
) ENGINE=InnoDB;
"""

with ENGINE.begin() as conn:
    for statement in SQL_CREATE_TABLAS.strip().split(";"):
        if statement.strip():
            conn.exec_driver_sql(statement)

print("Tablas listas: dim_moneda_cripto, hechos_mercado_cripto, noticias_cripto, tipo_cambio")

In [ ]:
def upsert_dataframe(df, table_name, engine, update_cols=None):
    """
    Inserta `df` en `table_name` con ON DUPLICATE KEY UPDATE. La tabla debe existir
    con su llave primaria ya definida (ver SQL_CREATE_TABLAS arriba).
    """
    if df.empty:
        print(f"[{table_name}] nada que cargar (DataFrame vacío)")
        return 0

    metadata = MetaData()
    tabla = Table(table_name, metadata, autoload_with=engine)
    pk_cols = {c.name for c in tabla.primary_key}

    registros = df.to_dict(orient="records")
    stmt = mysql_insert(tabla).values(registros)

    columnas_actualizar = update_cols or [c.name for c in tabla.columns if c.name not in pk_cols]
    stmt = stmt.on_duplicate_key_update({col: stmt.inserted[col] for col in columnas_actualizar})

    with engine.begin() as conn:
        conn.execute(stmt)

    print(f"[{table_name}] {len(registros)} filas cargadas (upsert)")
    return len(registros)

In [ ]:
upsert_dataframe(df_dim_moneda, "dim_moneda_cripto", ENGINE)
upsert_dataframe(df_hechos_mercado, "hechos_mercado_cripto", ENGINE)
upsert_dataframe(df_noticias, "noticias_cripto", ENGINE)
upsert_dataframe(df_tipo_cambio, "tipo_cambio", ENGINE)

## Verificar: las 4 tablas están separadas, pero se pueden unir con un JOIN

Este es el beneficio concreto de haber normalizado en 4 tablas: `dim_moneda_cripto` no
repite `name`/`symbol` en cada snapshot — solo se guarda una vez por moneda.

In [ ]:
consulta_join = """
    SELECT h.fecha_snapshot, d.name, d.symbol, h.price_usd, h.price_eur,
           h.change_24h_pct, h.is_alert, n.news_title
    FROM hechos_mercado_cripto h
    JOIN dim_moneda_cripto d ON h.coin_id = d.coin_id
    LEFT JOIN noticias_cripto n
        ON h.coin_id = n.coin_id AND h.fecha_snapshot = n.fecha_snapshot
    ORDER BY h.change_24h_pct DESC
"""

df_resultado = pd.read_sql(consulta_join, ENGINE)
df_resultado

## Dashboard (igual que el original, adaptado a las nuevas columnas)

In [ ]:
def create_dashboard(df):
    plt.figure(figsize=(14, 8))
    sns.set_theme(style="darkgrid")

    plot_df = df.head(15).copy()
    plot_df["direccion"] = plot_df["change_24h_pct"].apply(lambda x: "Sube" if x >= 0 else "Baja")

    ax = sns.barplot(
        x="change_24h_pct", y="name", data=plot_df,
        hue="direccion", palette={"Sube": "#27ae60", "Baja": "#c0392b"}, legend=False,
    )

    for i, (val, titulo) in enumerate(zip(plot_df["change_24h_pct"], plot_df["news_title"])):
        if pd.notna(titulo):
            ax.text(val, i, f" {titulo[:40]}...", va="center", fontsize=9, fontweight="bold")

    plt.title("Reporte de Mercado Cripto: Cambio % y Noticias Recientes", fontsize=16)
    plt.tight_layout()
    plt.show()


create_dashboard(df_resultado)

---
## Orquestador: correr todo el pipeline de una vez

Junta las 4 etapas de arriba (Extract → Transform → Extract → Transform → Load) con los
mismos banners, para correrlo completo sin ejecutar celda por celda.

In [ ]:
def run_pipeline(per_page=30):
    print("=" * 60); print("EXTRACT (1/2): mercado y tipo de cambio"); print("=" * 60)
    raw_fx = extract_exchange_rate("EUR")
    raw_market = extract_market_snapshot(per_page)
    eur_rate = raw_fx["rates"]["EUR"]

    print("=" * 60); print("TRANSFORM (1/2): limpiar mercado y detectar top movers"); print("=" * 60)
    fecha_snapshot = datetime.now(timezone.utc).replace(second=0, microsecond=0, tzinfo=None)
    df_dim, df_hechos, top_ids = transform_market_data(raw_market, eur_rate, fecha_snapshot)

    print("=" * 60); print("EXTRACT (2/2): noticias solo para top movers"); print("=" * 60)
    nombres = df_dim[df_dim["coin_id"].isin(top_ids)][["coin_id", "name"]]
    raw_news = []
    for _, fila in nombres.iterrows():
        titulo, link = extract_news_for_coin(fila["name"])
        raw_news.append({"coin_id": fila["coin_id"], "news_title": titulo, "news_url": link})
        time.sleep(0.2)

    print("=" * 60); print("TRANSFORM (2/2): tabla de noticias + validación"); print("=" * 60)
    df_news = transform_noticias(raw_news, fecha_snapshot)
    df_fx = transform_tipo_cambio(raw_fx)
    validar_calidad(df_hechos, df_fx)

    print("=" * 60); print("LOAD: 4 tablas"); print("=" * 60)
    upsert_dataframe(df_dim, "dim_moneda_cripto", ENGINE)
    upsert_dataframe(df_hechos, "hechos_mercado_cripto", ENGINE)
    upsert_dataframe(df_news, "noticias_cripto", ENGINE)
    upsert_dataframe(df_fx, "tipo_cambio", ENGINE)

    print("Pipeline completo.")
    return df_hechos, df_news


# run_pipeline()